# 📘 Notebook 12: Policy Gradients

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module D: Policy-Based Learning · Notebook 1 of 4 in this module · (12 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Give three reasons for learning a **policy directly** in place of action values
- Build a **policy network** that outputs a probability for each action, and sample actions from it
- State the **policy gradient** idea: make actions that were followed by high returns more probable
- Implement **REINFORCE** and train it on CartPole
- Explain why its gradient estimate has **high variance**, and reduce the variance with a **baseline**
- Compare policy-gradient and value-based methods: what each is good at

> **Prerequisites:** Notebook 04 (learning from complete episodes, returns computed backwards) and Notebook 08 (a network as function approximator). PyTorch as in the earlier notebooks.
>
> 🔭 **Why this notebook matters later:** every algorithm in the rest of this module, and most of the large reinforcement learning systems in use today, are descendants of the one written here. Notebook 13 adds a critic to it, and Notebook 14 turns the result into PPO.

> 📦 **Libraries used in this notebook.** **PyTorch**, **Gymnasium** and **imageio**. No GPU is needed. The two comparison cells train three agents each and take a few minutes.

In [ ]:
%pip install -q "gymnasium[classic-control]>=1.3" imageio imageio-ffmpeg torch

In [ ]:
import base64
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=30, width=420):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. A different road

Everything in Modules B and C followed one plan: learn how good each action is, then pick the best. The policy was never represented anywhere. It was whatever `argmax` made of the values.

That plan has three weak points.

- **Values can be harder than the policy.** Exercise 5 of Notebook 08 showed it: a rule as simple as "push towards the side the pole is falling to" balances CartPole, yet a linear model could not learn the *values* and failed completely.
- **`argmax` needs a short list of actions.** A steering wheel, a throttle or a robot joint can take any value in a range. There is nothing to take the maximum over. Notebook 15 is about such actions.
- **A greedy policy is deterministic.** In some games the best behaviour is random: in rock-paper-scissors any fixed choice is exploited at once. Notebook 17 comes back to this.

**Policy-based** methods take the direct road: represent the policy itself by a network, and adjust its parameters so that it earns more reward.

## 2. A policy network

### Formal definition
A **policy network** takes a state and returns a **probability for each action**, written $\pi(a \mid s; \boldsymbol\theta)$ with parameters $\boldsymbol\theta$ (theta). The last layer produces one unbounded number per action, called a **logit**, and the **softmax** function turns the logits into probabilities that are positive and sum to 1:

$$\pi(a \mid s) = \frac{e^{z_a}}{\sum_{b} e^{z_b}}$$

The agent **samples** its action from these probabilities. It does not take the most probable one.

In [ ]:
def make_policy(state_size, n_actions, hidden=64):
    return nn.Sequential(nn.Linear(state_size, hidden), nn.ReLU(),
                         nn.Linear(hidden, n_actions))          # outputs logits


env = gym.make("CartPole-v1")
torch.manual_seed(0)
policy = make_policy(4, 2)

state, info = env.reset(seed=0)
logits = policy(torch.as_tensor(state))
distribution = torch.distributions.Categorical(logits=logits)       # applies the softmax for us

print("logits:        ", logits.detach().numpy().round(3))
print("probabilities: ", distribution.probs.detach().numpy().round(3))
print("ten samples:   ", [int(distribution.sample()) for i in range(10)])

### Why it matters
Sampling gives **exploration for free**. There is no epsilon. While the policy is unsure, its probabilities are close to equal and it tries everything. As it learns, the probabilities sharpen and it explores less, by itself.

Small changes of the parameters also change the probabilities **smoothly**. With `argmax`, a tiny change in the values can flip the chosen action completely. This smoothness is what will let us use gradient ascent on the policy.

## 3. The policy gradient

### Intuition first
A cook tries small variations of a recipe. Each time the guests are pleased, the cook leans a little further in the direction of that variation, and each time they are not, a little away from it. Nobody tells the cook what the perfect recipe is. The feedback is only "better" or "worse", and the recipe drifts towards what works.

For our agent: play an episode with the current policy. Then, for every action that was taken, make it **more probable** if what followed was good and **less probable** if what followed was bad, in proportion to how good or bad.

### Formal definition
We want parameters $\boldsymbol\theta$ that maximise the expected return $J(\boldsymbol\theta)$ of the policy. The **policy gradient theorem** says in which direction to change them:

$$\nabla_{\boldsymbol\theta} J(\boldsymbol\theta) = \mathbb{E}_\pi\!\left[\,\sum_t \nabla_{\boldsymbol\theta} \log \pi(A_t \mid S_t; \boldsymbol\theta)\; G_t \right]$$

Two factors, for each step of an episode:

- $\nabla_{\boldsymbol\theta} \log \pi(A_t \mid S_t)$ is the direction in parameter space that makes the action **that was actually taken** more probable.
- $G_t$ is the return that followed it. It says **how far** to go in that direction, and with which sign.

The expectation is over episodes played with the policy, so one episode gives one **sample** of this gradient: a Monte Carlo estimate, in the spirit of Notebook 04.

The first factor can be checked in isolation. Take a "policy" with two logits and no state, and follow the gradient of $\log \pi(\text{action } 1)$ for a few steps.

In [ ]:
logits = torch.zeros(2, requires_grad=True)
print(f"start:        probability of action 1 = {torch.softmax(logits, dim=0)[1].item():.3f}")

for step in range(1, 4):
    log_probability = torch.log_softmax(logits, dim=0)[1]
    log_probability.backward()                       # gradient of log pi(action 1)
    with torch.no_grad():
        logits += 1.0 * logits.grad                  # one step uphill
        logits.grad.zero_()
    print(f"after step {step}: probability of action 1 = {torch.softmax(logits, dim=0)[1].item():.3f}")

Each step along the gradient of the log-probability raises the probability of that action, and the other action loses what it gains. Multiplying the step by a positive return reinforces the action. Multiplying by a negative return pushes the probability down.

## 4. REINFORCE

### Formal definition
**REINFORCE** (Ronald Williams, 1992) is the theorem turned into a loop:

1. Play **one complete episode** with the current policy, remembering the log-probability of every action taken and every reward.
2. Compute the return $G_t$ that followed each step, working backwards (Notebook 02).
3. Take one gradient step that increases $\sum_t \log \pi(A_t \mid S_t)\,G_t$.

PyTorch minimises, so the loss is that sum with a minus sign.

In [ ]:
def returns_to_go(rewards, gamma):
    """The discounted return that followed each step of an episode."""
    G, result = 0.0, []
    for reward in reversed(rewards):
        G = reward + gamma * G
        result.append(G)
    return torch.tensor(result[::-1], dtype=torch.float32)


def reinforce(env, episodes, baseline=True, learning_rate=1e-2, gamma=0.99, seed=0):
    """REINFORCE. Returns the policy network and the total reward of every episode."""
    torch.manual_seed(seed)
    policy = make_policy(env.observation_space.shape[0], env.action_space.n)
    optimiser = torch.optim.Adam(policy.parameters(), lr=learning_rate)
    history = []
    for episode in range(episodes):
        # 1. play one episode
        state, info = env.reset(seed=seed * 100000 + episode)
        log_probabilities, rewards = [], []
        terminated = truncated = False
        while not (terminated or truncated):
            distribution = torch.distributions.Categorical(logits=policy(torch.as_tensor(state, dtype=torch.float32)))
            action = distribution.sample()
            log_probabilities.append(distribution.log_prob(action))
            state, reward, terminated, truncated, info = env.step(int(action))
            rewards.append(reward)

        # 2. the return that followed each step
        G = returns_to_go(rewards, gamma)
        if baseline:
            G = (G - G.mean()) / (G.std() + 1e-8)        # Section 6 explains this line

        # 3. one gradient step
        loss = -(torch.stack(log_probabilities) * G).sum()
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        history.append(sum(rewards))
    return policy, np.array(history)

Notice what is **not** there: no replay buffer, no target network, no epsilon, no `max`. The whole algorithm is one episode, one sum, one gradient step.

We train three agents with different seeds for 300 episodes. This takes about a minute.

In [ ]:
runs = [reinforce(env, 300, seed=seed) for seed in range(3)]


def smooth(values, window=20):
    return np.convolve(values, np.ones(window) / window, mode="valid")


plt.figure(figsize=(8, 4))
for seed, (policy, history) in enumerate(runs):
    plt.plot(smooth(history), label=f"seed {seed}")
plt.axhline(500, color="gray", linestyle=":", label="perfect score")
plt.title("REINFORCE on CartPole")
plt.xlabel("episode"); plt.ylabel("steps the pole stayed up (average of 20)"); plt.grid(True); plt.legend()
plt.show()

for seed, (policy, history) in enumerate(runs):
    print(f"seed {seed}: best 20-episode average {smooth(history).max():4.0f},  last 20 episodes {history[-20:].mean():4.0f}")

All three agents get close to the perfect score of 500 at some point, with an algorithm of a dozen lines. DQN, with its buffer and its second network, did not reach that level in Notebook 09. The comparison is not quite fair, though: DQN was given 20,000 steps of experience there, and 300 episodes of REINFORCE amount to several times as many. Policy gradients are simple, and they are hungry for experience. The curves are also not smooth, and a run can fall back after it has done well. Section 6 looks at why.

In [ ]:
def record(env, policy, seed=0, sample=True):
    state, info = env.reset(seed=seed)
    frames = [env.render()]
    terminated = truncated = False
    while not (terminated or truncated):
        with torch.no_grad():
            logits = policy(torch.as_tensor(state, dtype=torch.float32))
        action = int(torch.distributions.Categorical(logits=logits).sample()) if sample else int(logits.argmax())
        state, reward, terminated, truncated, info = env.step(action)
        frames.append(env.render())
    return frames


best = int(np.argmax([history[-20:].mean() for policy, history in runs]))
frames = record(gym.make("CartPole-v1", render_mode="rgb_array"), runs[best][0], seed=1)
print(f"seed {best}: the pole stayed up for {len(frames) - 1} steps")
show_video(frames[::3], fps=20)

## 5. What the policy has learned

A trained policy is a function from states to probabilities, and we can look at it. The plot shows the probability of "push right" as the pole's angle varies, with the other three state variables set to zero.

In [ ]:
policy = runs[best][0]
angles = np.linspace(-0.2, 0.2, 81)
states = torch.zeros(len(angles), 4)
states[:, 2] = torch.as_tensor(angles, dtype=torch.float32)

with torch.no_grad():
    push_right = torch.softmax(policy(states), dim=1)[:, 1].numpy()

plt.figure(figsize=(7, 4))
plt.plot(np.degrees(angles), push_right)
plt.axhline(0.5, color="gray", linestyle=":")
plt.title("The learned policy")
plt.xlabel("angle of the pole in degrees (positive: leaning right)"); plt.ylabel("probability of pushing right")
plt.grid(True)
plt.show()

for degrees in (-8, -2, 0, 2, 8):
    index = int(np.argmin(np.abs(np.degrees(angles) - degrees)))
    print(f"pole at {degrees:+d} degrees: probability of pushing right {push_right[index]:.2f}")

When the pole leans clearly to one side, the policy pushes that way with high probability. Around the upright position the probability is close to one half: there the choice matters little, and the policy stays undecided. A value-based agent would have to pick one action even when the two values differ by a hair.

## 6. The variance problem and the baseline

### Intuition first
In CartPole every reward is +1, so **every** return is positive. Read the update rule again with that in mind: every action that was taken is made *more* probable, the good ones a lot and the poor ones a little. The policy improves only because the good actions are pushed up harder than the bad ones. That is like marking an exam where the worst answer gets 60 points and the best gets 65: the ranking is in there, but buried under a large common amount that carries no information.

Worse, the return of an episode depends on a long chain of random action choices, so the same action in the same state is followed by very different returns from one episode to the next. The gradient estimate is correct on average and **very noisy** in any single episode. Statisticians say it has high **variance**.

### Formal definition
A **baseline** $b$ is a number subtracted from the return:

$$\nabla_{\boldsymbol\theta} J(\boldsymbol\theta) = \mathbb{E}_\pi\!\left[\,\sum_t \nabla_{\boldsymbol\theta} \log \pi(A_t \mid S_t; \boldsymbol\theta)\,\big(G_t - b\big)\right]$$

As long as $b$ does not depend on the action, the **expected** gradient is unchanged, and a well-chosen $b$ makes the individual samples far less noisy. Actions followed by a better-than-usual return now get a positive weight and are reinforced. Actions followed by a worse-than-usual return get a **negative** weight and are discouraged.

The line marked in `reinforce` **standardises** the returns of the episode. It does two things. It subtracts their mean, which is the simplest possible baseline. And it divides by their standard deviation, so that the size of an update does not depend on the scale of the rewards or on the length of the episode.

Does it matter? The same three seeds, without that line:

In [ ]:
runs_plain = [reinforce(env, 300, baseline=False, seed=seed) for seed in range(3)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, title, group in ((axes[0], "with baseline", runs), (axes[1], "without baseline", runs_plain)):
    for seed, (policy, history) in enumerate(group):
        ax.plot(smooth(history), label=f"seed {seed}")
    ax.axhline(500, color="gray", linestyle=":")
    ax.set_title(f"REINFORCE {title}")
    ax.set_xlabel("episode"); ax.grid(True); ax.legend()
axes[0].set_ylabel("steps the pole stayed up (average of 20)")
plt.show()

for name, group in (("with baseline", runs), ("without baseline", runs_plain)):
    averages = [history.mean() for policy, history in group]
    print(f"{name:17s} average over all 300 episodes, per seed: " + "  ".join(f"{a:4.0f}" for a in averages))

Averaged over the three seeds, the agents without the baseline collect clearly less reward, and they reach high scores later or not at all. Compare seed by seed as well: with so much noise, a single run without the baseline can still come out ahead of its counterpart. One line of code, which changes nothing about what is being optimised, shifts the odds.

Both halves of that line contribute, and a constant subtracted from every return is a crude baseline. A good baseline should depend on the **state**: a return of 50 is disappointing from a state where 200 is normal, and excellent from a state where the pole is about to fall. Notebook 13 builds such a baseline and measures the noise of the gradient directly.

> ⚠️ **Common pitfalls**
>
> - **Taking `argmax` during training.** The policy must **sample**. The gradient formula assumes the actions were drawn from $\pi$, and without sampling there is no exploration.
> - **Forgetting the minus sign.** We want to *maximise* $\log \pi \cdot G$, and optimisers minimise.
> - **Reusing old episodes.** The expectation in the theorem is over the **current** policy. Data from an older policy gives a wrong gradient. This is what "on-policy" means, and it is why there is no replay buffer.
> - **A baseline that depends on the action.** That changes the expected gradient and biases the learning.

## 7. Two families

| | Value-based (DQN) | Policy-based (REINFORCE) |
|---|---|---|
| What the network outputs | a value for each action | a probability for each action |
| How it acts | greedily, plus epsilon for exploration | by sampling from its own probabilities |
| Learns from | single steps, bootstrapped | complete episodes |
| Old experience | reused many times (off-policy) | thrown away after one update (on-policy) |
| Experience needed | less | more |
| Continuous actions | no | yes (Notebook 15) |
| Typical difficulty | unstable values, overestimation | noisy gradients |

REINFORCE has the two limits of Monte Carlo learning that Notebook 04 listed: it must wait for the end of an episode, and its estimates are noisy. Notebook 05 answered those limits for value learning with temporal differences. The next notebook does the same here: it trains a value network alongside the policy and uses it as a much better baseline. The combination is called **actor-critic**.

---
## ✏️ Exercises

The exercises use `env`, `make_policy`, `returns_to_go`, `reinforce` and `smooth` from above.

### Exercise 1 (Softmax by hand)
A policy network outputs the logits `[2.0, 1.0, 0.0]` for three actions. Compute the three probabilities with NumPy, applying the softmax formula yourself, and check with `torch.softmax`. Then add 10 to every logit and compute them again. What do you observe, and why?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
logits = np.array([2.0, 1.0, 0.0])
by_hand = np.exp(logits) / np.exp(logits).sum()
print("by hand: ", by_hand.round(3))
print("PyTorch: ", torch.softmax(torch.tensor(logits), dim=0).numpy().round(3))

shifted = logits + 10
print("shifted: ", (np.exp(shifted) / np.exp(shifted).sum()).round(3))
```

*Adding the same number to every logit multiplies numerator and denominator by the same factor, so the probabilities do not change. Only the **differences** between logits matter. A difference of 1 between two logits means a ratio of e, about 2.7, between the two probabilities.*
</details>

### Exercise 2 (Which actions get reinforced?)
An episode of three steps has the rewards `[1, 1, 1]`. With gamma = 1, compute the returns to go. Then compute the weights after subtracting their mean. Which steps would be reinforced and which discouraged by one update, with and without the baseline? Does the result make sense for CartPole?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
G = returns_to_go([1, 1, 1], gamma=1.0)
print("returns to go:      ", G.numpy())
print("minus their mean:   ", (G - G.mean()).numpy())
```

*Without a baseline all three actions are reinforced, the first one most. With the baseline the first action gets a positive weight and the last a negative one. For CartPole that is sensible: an action taken early in a long episode was followed by many more steps of balancing than the action taken just before the pole fell, and the last actions of an episode are the likeliest to have caused its end.*
</details>

### Exercise 3 (The whole-episode return)
The earliest form of the algorithm weighted **every** action of an episode by the return of the whole episode, $G_0$, and not by the return that followed that action. Write `reinforce_whole_episode` by changing the returns in a copy of `reinforce` (use no baseline, so that the comparison is with `baseline=False`), train it for 300 episodes with seeds 0 to 2, and compare the average reward over all episodes with `runs_plain`. Why is the return to go the better weight?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def reinforce_whole_episode(env, episodes, learning_rate=1e-2, gamma=0.99, seed=0):
    torch.manual_seed(seed)
    policy = make_policy(env.observation_space.shape[0], env.action_space.n)
    optimiser = torch.optim.Adam(policy.parameters(), lr=learning_rate)
    history = []
    for episode in range(episodes):
        state, info = env.reset(seed=seed * 100000 + episode)
        log_probabilities, rewards = [], []
        terminated = truncated = False
        while not (terminated or truncated):
            distribution = torch.distributions.Categorical(logits=policy(torch.as_tensor(state, dtype=torch.float32)))
            action = distribution.sample()
            log_probabilities.append(distribution.log_prob(action))
            state, reward, terminated, truncated, info = env.step(int(action))
            rewards.append(reward)
        G = returns_to_go(rewards, gamma)
        G = torch.full_like(G, float(G[0]))               # the same weight for every step
        loss = -(torch.stack(log_probabilities) * G).sum()
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        history.append(sum(rewards))
    return policy, np.array(history)

whole = [reinforce_whole_episode(env, 300, seed=seed)[1].mean() for seed in range(3)]
to_go = [history.mean() for policy, history in runs_plain]
print("whole-episode return: average reward per seed " + "  ".join(f"{a:4.0f}" for a in whole))
print("return to go:         average reward per seed " + "  ".join(f"{a:4.0f}" for a in to_go))
```

*An action cannot influence rewards that were received **before** it was taken. The whole-episode return includes those earlier rewards, which are pure noise as far as this action is concerned. Leaving them out removes noise without changing the expected gradient. It is the first and cheapest variance reduction.*
</details>

### Exercise 4 (The learning rate)
Train REINFORCE (with baseline) for 300 episodes with learning rates of 0.001 and 0.05, two seeds each, and compare the average of the last 50 episodes with the notebook's 0.01. What happens when the steps are too large? Remember that each update is based on a single noisy episode.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for learning_rate in (0.001, 0.05):
    for seed in range(2):
        policy, history = reinforce(env, 300, learning_rate=learning_rate, seed=seed)
        print(f"learning rate {learning_rate:<6} seed {seed}: last 50 episodes {history[-50:].mean():4.0f},  "
              f"best 20-episode average {smooth(history).max():4.0f}")
for seed in range(2):
    history = runs[seed][1]
    print(f"learning rate 0.01   seed {seed}: last 50 episodes {history[-50:].mean():4.0f},  "
          f"best 20-episode average {smooth(history).max():4.0f}")
```

*With a very small learning rate the policy improves steadily but has not got far in 300 episodes. With a very large one it can improve quickly and then destroy itself: one unlucky episode produces a large step in a bad direction, the policy becomes poor, and a poor policy only generates poor episodes to learn from. Compare the "best" and "last" columns. A policy-gradient step that is too large can be impossible to recover from, and limiting the size of each step is the central idea of PPO in Notebook 14.*
</details>

### Exercise 5 (Several episodes per update (a little harder))
One way to reduce the noise is to average the gradient over several episodes before taking a step. Write `reinforce_batch(env, updates, batch, ...)` that plays `batch` episodes with the current policy, adds up their losses, divides by `batch`, and then takes one optimiser step. Compare, for seeds 0 and 1, 300 updates of 1 episode with 60 updates of 5 episodes, so that both use 300 episodes in total. Print the average reward of the last 50 episodes for each. Which uses its experience better here?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def reinforce_batch(env, updates, batch, learning_rate=1e-2, gamma=0.99, seed=0):
    torch.manual_seed(seed)
    policy = make_policy(env.observation_space.shape[0], env.action_space.n)
    optimiser = torch.optim.Adam(policy.parameters(), lr=learning_rate)
    history = []
    for update in range(updates):
        loss = 0.0
        for episode in range(batch):
            state, info = env.reset(seed=seed * 100000 + len(history))
            log_probabilities, rewards = [], []
            terminated = truncated = False
            while not (terminated or truncated):
                distribution = torch.distributions.Categorical(logits=policy(torch.as_tensor(state, dtype=torch.float32)))
                action = distribution.sample()
                log_probabilities.append(distribution.log_prob(action))
                state, reward, terminated, truncated, info = env.step(int(action))
                rewards.append(reward)
            G = returns_to_go(rewards, gamma)
            G = (G - G.mean()) / (G.std() + 1e-8)
            loss = loss - (torch.stack(log_probabilities) * G).sum()
            history.append(sum(rewards))
        optimiser.zero_grad()
        (loss / batch).backward()
        optimiser.step()
    return policy, np.array(history)

for seed in range(2):
    single = runs[seed][1]
    batched = reinforce_batch(env, updates=60, batch=5, seed=seed)[1]
    print(f"seed {seed}: 300 updates of 1 episode: last 50 episodes {single[-50:].mean():4.0f}    "
          f"60 updates of 5 episodes: last 50 episodes {batched[-50:].mean():4.0f}")
```

*Averaging five episodes gives a less noisy gradient, but only a fifth as many steps. On a problem as easy as CartPole the many noisy steps usually win. The trade-off between the number of updates and the quality of each one is present in every policy-gradient method, and Notebook 14 shows how PPO gets several good updates out of one batch of experience.*
</details>

## 🔑 Key takeaways

- **Policy-based** methods represent the policy by a network that outputs **action probabilities** and adjust it directly.
- Sampling from the policy gives **exploration without epsilon**, and makes random policies possible.
- The **policy gradient**: raise the log-probability of each action taken, in proportion to the **return that followed**.
- **REINFORCE** is that idea with complete episodes: play, compute returns to go, take one gradient step.
- Its gradient estimate is **noisy**. Subtracting a **baseline** reduces the variance without changing the expected gradient.
- Policy gradients are **on-policy**: each batch of experience is used once and then discarded.

---
**Next:** *Notebook 13: Actor-Critic Methods*, where a second network learns to judge the first one's actions.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*